# Out-of-sample validation 02

Freeze the Phase 1 sizing estimate using information available through November 4, 2024, then predict TAN's November 6 market-adjusted move:

\[
\widehat{r}^{MA}_{TAN,Nov\ 6} = \Delta E \times (1 - p_{Nov\ 4}).
\]

`ΔE` is the pre-specified event-window high-signal estimate from Phase 1. No November 5 or November 6 outcomes enter the prediction.

In [2]:
from pathlib import Path

import pandas as pd

PHASE1_RESULTS_PATH = Path("data/estimatedelata_01_final_comparison.csv")
DAILY_DATA_PATH = Path("data/daily_merged_tan_spy_icln_polymarket_2024.csv")
PREDICTION_PATH = Path("data/outofsamplevalidation02_nov6_prediction.csv")
CUTOFF_DATE = pd.Timestamp("2024-11-04")
TARGET_DATE = pd.Timestamp("2024-11-06")

In [3]:
phase1_results = pd.read_csv(PHASE1_RESULTS_PATH)
daily = pd.read_csv(DAILY_DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

# Freeze inputs at the end of November 4.
phase1_daily = daily.loc[:CUTOFF_DATE].copy()
assert phase1_daily.index.max() == CUTOFF_DATE
assert TARGET_DATE not in phase1_daily.index

sizing_row = phase1_results.loc[
    phase1_results["role"].eq("Headline / sizing")
]
if len(sizing_row) != 1:
    raise RuntimeError(f"Expected one Phase 1 sizing estimate, found {len(sizing_row)}.")

delta_E = float(sizing_row.iloc[0]["delta_E"])
delta_E_se = float(sizing_row.iloc[0]["standard_error"])
p_nov4 = float(phase1_daily.loc[CUTOFF_DATE, "p"])

{
    "phase1_cutoff": CUTOFF_DATE.date().isoformat(),
    "delta_E_frozen": delta_E,
    "delta_E_se": delta_E_se,
    "p_Nov4": p_nov4,
}

{'phase1_cutoff': '2024-11-04',
 'delta_E_frozen': -0.23420197593491,
 'delta_E_se': 0.1127156986524125,
 'p_Nov4': 0.5855}

In [4]:
remaining_probability_move = 1.0 - p_nov4
predicted_market_adjusted_move = delta_E * remaining_probability_move
predicted_se_from_delta_E = abs(remaining_probability_move) * delta_E_se

prediction = pd.DataFrame(
    {
        "phase1_cutoff": [CUTOFF_DATE.date().isoformat()],
        "target_date": [TARGET_DATE.date().isoformat()],
        "delta_E_frozen": [delta_E],
        "delta_E_se": [delta_E_se],
        "p_Nov4": [p_nov4],
        "remaining_probability_move": [remaining_probability_move],
        "predicted_TAN_market_adjusted_move": [predicted_market_adjusted_move],
        "predicted_se_from_delta_E_only": [predicted_se_from_delta_E],
    }
)
prediction.to_csv(PREDICTION_PATH, index=False)

print(f"Frozen ΔE: {delta_E:.6f}")
print(f"p_Nov4: {p_nov4:.6f}")
print(f"1 − p_Nov4: {remaining_probability_move:.6f}")
print(
    "Predicted Nov 6 TAN market-adjusted move: "
    f"{predicted_market_adjusted_move:.6f} "
    f"({predicted_market_adjusted_move:.2%})"
)
print(f"Saved prediction to {PREDICTION_PATH.resolve()}")

prediction

Frozen ΔE: -0.234202
p_Nov4: 0.585500
1 − p_Nov4: 0.414500
Predicted Nov 6 TAN market-adjusted move: -0.097077 (-9.71%)
Saved prediction to /Users/nick/Desktop/Prediction Markets/data/outofsamplevalidation02_nov6_prediction.csv


,phase1_cutoff,target_date,delta_E_frozen,delta_E_se,p_Nov4,remaining_probability_move,predicted_TAN_market_adjusted_move,predicted_se_from_delta_E_only
0,2024-11-04,2024-11-06,-0.234202,0.112716,0.5855,0.4145,-0.097077,0.046721


## Frozen prediction

- Phase 1 cutoff: **November 4, 2024**
- Frozen event-window ΔE: **−0.234202**
- November 4 probability: **0.5855**
- Remaining probability move to resolution: **0.4145**
- **Predicted November 6 TAN market-adjusted move: −0.097077 (−9.71%)**
- SE propagated from ΔE only: **0.046721**

The propagated SE treats `p_Nov4` as fixed and therefore does not include prediction-market measurement uncertainty.

## Realized out-of-sample comparison

Realized moves use adjusted-close returns and subtract SPY's corresponding return from TAN's return. The November 6 measure is close-to-close. The cumulative November 6–12 measure compounds each ETF from the November 5 close through the November 12 close, allowing for a delayed equity response.

For each horizon:

\[
\text{estimation error} = \frac{\text{realized} - \text{predicted}}{\text{predicted}}.
\]

Because the prediction is negative, a positive error means the realized decline was larger in magnitude than predicted.

In [ ]:
ETF_DATA_PATH = Path("data/tan_spy_icln_daily_2016_2024.csv")
VALIDATION_RESULTS_PATH = Path("data/outofsamplevalidation02_realized_comparison.csv")
LAG_END_DATE = pd.Timestamp("2024-11-12")

etf_prices = pd.read_csv(ETF_DATA_PATH, parse_dates=["Date"])
adjusted_close = (
    etf_prices.pivot(index="Date", columns="Ticker", values="Adj Close")
    .sort_index()
)
etf_returns = adjusted_close.pct_change(fill_method=None)

# Nov 6 close-to-close market-adjusted return.
nov6_tan_return = etf_returns.loc[TARGET_DATE, "TAN"]
nov6_spy_return = etf_returns.loc[TARGET_DATE, "SPY"]
realized_nov6 = nov6_tan_return - nov6_spy_return

# Compound Nov 6–12 returns (Nov 5 close through Nov 12 close).
lag_window_returns = etf_returns.loc[TARGET_DATE:LAG_END_DATE, ["TAN", "SPY"]]
cumulative_returns = (1 + lag_window_returns).prod() - 1
realized_nov6_12 = cumulative_returns["TAN"] - cumulative_returns["SPY"]

assert lag_window_returns.index.min() == TARGET_DATE
assert lag_window_returns.index.max() == LAG_END_DATE
assert lag_window_returns.notna().all().all()

realized_nov6, realized_nov6_12

In [ ]:
validation = pd.DataFrame(
    [
        {
            "horizon": "Nov 6",
            "predicted_market_adjusted_move": predicted_market_adjusted_move,
            "realized_TAN_return": nov6_tan_return,
            "realized_SPY_return": nov6_spy_return,
            "realized_market_adjusted_move": realized_nov6,
        },
        {
            "horizon": "Nov 6–12 cumulative",
            "predicted_market_adjusted_move": predicted_market_adjusted_move,
            "realized_TAN_return": cumulative_returns["TAN"],
            "realized_SPY_return": cumulative_returns["SPY"],
            "realized_market_adjusted_move": realized_nov6_12,
        },
    ]
)
validation["estimation_error"] = (
    validation["realized_market_adjusted_move"]
    - validation["predicted_market_adjusted_move"]
) / validation["predicted_market_adjusted_move"]

validation.to_csv(VALIDATION_RESULTS_PATH, index=False)

for row in validation.itertuples(index=False):
    print(
        f"{row.horizon}: predicted={row.predicted_market_adjusted_move:.2%}, "
        f"realized={row.realized_market_adjusted_move:.2%}, "
        f"estimation error={row.estimation_error:.2%}"
    )
print(f"Saved comparison to {VALIDATION_RESULTS_PATH.resolve()}")

validation

## Validation results

- Predicted market-adjusted move: **−9.71%**
- Realized November 6 market-adjusted move: **−13.35%**
  - Estimation error: **37.52%**
- Realized cumulative November 6–12 market-adjusted move: **−20.42%**
  - Estimation error: **110.36%**

The sign was predicted correctly. The one-day decline was 37.52% larger in magnitude than predicted, while the cumulative decline was more than twice the predicted magnitude, consistent with substantial lagged equity adjustment.

## Final deliverable

The headline realized ΔE error uses the full five-trading-day November 6–12 response window, because that window captures the documented equity lag. The one-day result is retained as the timing comparison rather than as a second headline error.

In [ ]:
FINAL_DELIVERABLE_PATH = Path("data/outofsamplevalidation02_final_deliverable.csv")

one_day_row = validation.loc[validation["horizon"].eq("Nov 6")].iloc[0]
five_day_row = validation.loc[validation["horizon"].eq("Nov 6–12 cumulative")].iloc[0]

realized_delta_E_error = five_day_row["estimation_error"]
lag_increment = (
    five_day_row["realized_market_adjusted_move"]
    - one_day_row["realized_market_adjusted_move"]
)
five_to_one_day_ratio = abs(
    five_day_row["realized_market_adjusted_move"]
    / one_day_row["realized_market_adjusted_move"]
)

final_deliverable = pd.DataFrame(
    {
        "realized_delta_E_error_5day": [realized_delta_E_error],
        "predicted_move": [predicted_market_adjusted_move],
        "realized_1day_move": [one_day_row["realized_market_adjusted_move"]],
        "realized_5day_move": [five_day_row["realized_market_adjusted_move"]],
        "additional_lagged_move": [lag_increment],
        "five_to_one_day_magnitude_ratio": [five_to_one_day_ratio],
    }
)
final_deliverable.to_csv(FINAL_DELIVERABLE_PATH, index=False)

print(f"Realized ΔE error (5-day): {realized_delta_E_error:.2%}")
print(
    f"1-day vs 5-day realized: "
    f"{one_day_row['realized_market_adjusted_move']:.2%} vs "
    f"{five_day_row['realized_market_adjusted_move']:.2%}"
)
print(f"Additional lagged move: {lag_increment:.2%}")
print(f"5-day / 1-day magnitude: {five_to_one_day_ratio:.2f}×")
print(f"Saved final deliverable to {FINAL_DELIVERABLE_PATH.resolve()}")

final_deliverable

# Final result

**Realized ΔE error: 110.36%** using the five-day response window.

Lag comparison:

- One-day realized market-adjusted move: **−13.35%**
- Five-day realized market-adjusted move: **−20.42%**
- Additional move after day one: **−7.07 percentage points**
- Five-day magnitude relative to day one: **1.53×**

The larger five-day response shows that a substantial share of the equity adjustment occurred after November 6.